# Notebook 05 — Alignment Statistics: MAPQ, Flagstat, and Coverage

**Module:** 09 — Genomics and NGS  
**Notebook:** 05 of 16  
**Time estimate:** 75 minutes

> After alignment, you need summary statistics before any downstream analysis.
> `samtools flagstat` and `samtools coverage` are the first things you run on any BAM.

---
## Step 1 — Motivation

A BAM file can contain hundreds of millions of reads. Alignment statistics compress
this into diagnostic numbers: what fraction aligned? What is the mapping quality
distribution? What is the coverage depth? These numbers are quality gates — a run
with 50% alignment rate or 5× coverage instead of 30× has failed and should not
proceed to variant calling.

---
## Step 2 — Intuition

**`samtools flagstat` output (for each line — what it means):**
- `X + Y in total` — X primary reads + Y secondary/supplementary
- `X mapped (Y%)` — fraction of reads with a primary alignment
- `X properly paired (Y%)` — both mates mapped at expected distance/orientation
- `X singletons` — one mate mapped, other did not

**Typical QC thresholds for WGS:**
- Mapping rate > 95% (human)
- Properly paired > 90%
- Duplicate rate < 20%
- Mean coverage > 25× after duplicate removal

**Coverage uniformity matters as much as mean coverage.** A mean of 30× with very
high variance (GC bias) leaves many positions at 0× — no calls possible there.

---
## Step 3 — Biological Background

**Sources of low coverage:**
1. **GC bias:** PCR efficiency drops at extreme GC content. Regions with >70% or
   <30% GC are under-represented in Illumina libraries.
2. **Repetitive regions:** Multi-mapping reads (MAPQ=0) are excluded from most
   analyses. Centromeres, telomeres, and segmental duplications are effectively
   invisible to short-read sequencing.
3. **Library complexity:** If the original DNA was degraded or in low quantity,
   PCR amplification produces many identical copies → high duplicate rate → fewer
   informative reads.

**Theoretical vs. effective coverage:**
- Theoretical: all reads counted (C = N×L/G)
- Effective: after removing duplicates, multi-mappers, QC-failed reads

---
## Step 4 — Mathematical Explanation

**Coverage from a pileup:**  
For each reference position $p$, coverage depth $c_p$ = number of reads whose
alignment spans position $p$:
$$c_p = |\{r : \text{ref\_start}(r) \leq p \leq \text{ref\_end}(r)\}|$$

**Mean coverage:**
$$\bar{C} = \frac{1}{|G|} \sum_{p \in G} c_p$$

**Breadth of coverage at threshold $t$:**
$$\text{breadth}(t) = \frac{|\{p : c_p \geq t\}|}{|G|}$$

**Coefficient of variation (CV) of coverage:**
$$\text{CV} = \frac{\sigma_C}{\bar{C}}$$
Lower CV = more uniform coverage. GC bias inflates CV significantly.

In [ ]:
# Step 6 — Python: flagstat and coverage analysis from scratch
import numpy as np
from dataclasses import dataclass, field
from collections import defaultdict
import re


# Reuse SamRecord definition from NB04
@dataclass
class CigarOp:
    length: int
    op: str
    CONSUMES_QUERY = set('MISH=X')
    CONSUMES_REF = set('MDNX=')
    @property
    def consumes_ref(self) -> bool:
        return self.op in self.CONSUMES_REF


def parse_cigar(cigar: str) -> list:
    if cigar == '*': return []
    return [CigarOp(int(l), op) for l, op in re.findall(r'(\d+)([MIDNSHP=X])', cigar)]


@dataclass
class SamRecord:
    qname: str
    flag: int
    rname: str
    pos: int
    mapq: int
    cigar: str
    seq: str
    qual: str

    @property
    def ref_end(self) -> int:
        ops = parse_cigar(self.cigar)
        return self.pos + sum(op.length for op in ops if op.consumes_ref) - 1

    @property
    def is_mapped(self) -> bool: return not (self.flag & 0x4)
    @property
    def is_duplicate(self) -> bool: return bool(self.flag & 0x400)
    @property
    def is_secondary(self) -> bool: return bool(self.flag & 0x100)
    @property
    def is_supplementary(self) -> bool: return bool(self.flag & 0x800)
    @property
    def is_properly_paired(self) -> bool: return bool(self.flag & 0x2)
    @property
    def is_paired(self) -> bool: return bool(self.flag & 0x1)
    @property
    def is_singleton(self) -> bool:
        return self.is_paired and self.is_mapped and bool(self.flag & 0x8)


def flagstat(records: list[SamRecord]) -> dict:
    """Compute samtools flagstat-equivalent statistics."""
    primary = [r for r in records if not r.is_secondary and not r.is_supplementary]
    secondary = [r for r in records if r.is_secondary]
    supplementary = [r for r in records if r.is_supplementary]

    mapped_primary = [r for r in primary if r.is_mapped]
    properly_paired = [r for r in primary if r.is_properly_paired]
    duplicates = [r for r in primary if r.is_duplicate]
    singletons = [r for r in primary if r.is_singleton]

    n = len(records)
    n_primary = len(primary)

    return {
        'total': n,
        'primary': n_primary,
        'secondary': len(secondary),
        'supplementary': len(supplementary),
        'duplicates': len(duplicates),
        'duplicate_rate': len(duplicates) / n_primary if n_primary else 0,
        'mapped': len(mapped_primary),
        'mapping_rate': len(mapped_primary) / n_primary if n_primary else 0,
        'properly_paired': len(properly_paired),
        'properly_paired_rate': len(properly_paired) / n_primary if n_primary else 0,
        'singletons': len(singletons),
    }


def compute_coverage_vector(records: list[SamRecord], ref_length: int,
                             min_mapq: int = 0, exclude_dups: bool = False) -> np.ndarray:
    """Build a per-position coverage vector."""
    cov = np.zeros(ref_length + 1, dtype=np.int32)
    for r in records:
        if not r.is_mapped: continue
        if r.is_secondary or r.is_supplementary: continue
        if exclude_dups and r.is_duplicate: continue
        if r.mapq < min_mapq: continue
        start = r.pos - 1  # convert 1-based SAM to 0-based
        end = r.ref_end     # ref_end is 1-based inclusive
        if start < 0 or end > ref_length: continue
        cov[start:end] += 1
    return cov[:ref_length]


def coverage_stats(cov: np.ndarray) -> dict:
    return {
        'mean': float(cov.mean()),
        'median': float(np.median(cov)),
        'std': float(cov.std()),
        'cv': float(cov.std() / cov.mean()) if cov.mean() > 0 else float('inf'),
        'min': int(cov.min()),
        'max': int(cov.max()),
        'breadth_1x': float((cov >= 1).mean()),
        'breadth_10x': float((cov >= 10).mean()),
        'breadth_20x': float((cov >= 20).mean()),
        'breadth_30x': float((cov >= 30).mean()),
    }


# Simulate a BAM-like dataset
def simulate_alignment_records(
    n_reads: int = 1000,
    ref_length: int = 10000,
    read_length: int = 150,
    dup_rate: float = 0.10,
    unmapped_rate: float = 0.02,
    seed: int = 42
) -> list[SamRecord]:
    rng = np.random.default_rng(seed)
    records = []
    for i in range(n_reads):
        is_dup = rng.random() < dup_rate
        is_unmapped = rng.random() < unmapped_rate
        mapq = 0 if is_unmapped else int(rng.choice(
            [0, 20, 30, 40, 50, 60],
            p=[0.03, 0.05, 0.07, 0.15, 0.20, 0.50]
        ))
        flag = 0
        if not is_unmapped: flag |= 0x2 | 0x1  # properly paired
        if is_unmapped: flag |= 0x4
        if is_dup: flag |= 0x400
        pos = int(rng.integers(1, ref_length - read_length + 2)) if not is_unmapped else 0
        records.append(SamRecord(
            qname=f'read_{i:06d}',
            flag=flag,
            rname='chr1' if not is_unmapped else '*',
            pos=pos,
            mapq=mapq,
            cigar=f'{read_length}M' if not is_unmapped else '*',
            seq='A' * read_length,
            qual='I' * read_length,
        ))
    return records


records = simulate_alignment_records(n_reads=3000, ref_length=10000)
stats = flagstat(records)
cov = compute_coverage_vector(records, ref_length=10000, min_mapq=20, exclude_dups=True)
cov_all = compute_coverage_vector(records, ref_length=10000)

print('=== samtools flagstat ===\n')
print(f"{stats['total']} in total ({stats['secondary']} secondary + {stats['supplementary']} supplementary)")
print(f"{stats['mapped']} mapped ({stats['mapping_rate']*100:.1f}%)")
print(f"{stats['properly_paired']} properly paired ({stats['properly_paired_rate']*100:.1f}%)")
print(f"{stats['duplicates']} duplicates ({stats['duplicate_rate']*100:.1f}%)")
print(f"{stats['singletons']} singletons")
print()
print('=== Coverage statistics (after MAPQ≥20 filter, duplicates excluded) ===')
cv_stats = coverage_stats(cov)
for k, v in cv_stats.items():
    print(f"  {k:<20}: {v:.3f}")

In [ ]:
# Step 7 — Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Coverage depth across the reference
ax = axes[0, 0]
ax.fill_between(range(len(cov)), cov, alpha=0.6, color='steelblue', label='MAPQ≥20, dups removed')
ax.fill_between(range(len(cov_all)), cov_all, alpha=0.3, color='gray', label='All reads')
ax.axhline(cov.mean(), color='red', ls='--', lw=1.5, label=f'Mean={cov.mean():.1f}×')
ax.axhline(20, color='orange', ls=':', lw=1, label='20× threshold')
ax.set_xlabel('Reference position (bp)')
ax.set_ylabel('Coverage depth')
ax.set_title('A. Coverage depth across reference')
ax.legend(fontsize=8)

# Panel B: MAPQ distribution
ax2 = axes[0, 1]
mapq_vals = [r.mapq for r in records if r.is_mapped and not r.is_secondary]
ax2.hist(mapq_vals, bins=62, range=(-1, 61), color='tomato', edgecolor='none', alpha=0.8)
ax2.axvline(20, color='black', ls='--', label='MAPQ=20 filter')
ax2.set_xlabel('MAPQ')
ax2.set_ylabel('Number of reads')
ax2.set_title('B. MAPQ distribution')
ax2.legend(fontsize=8)

# Panel C: Breadth of coverage at different thresholds
ax3 = axes[1, 0]
thresholds = range(1, 61)
breadths = [(cov >= t).mean() * 100 for t in thresholds]
ax3.plot(thresholds, breadths, 'b-', lw=2)
for t_mark in [10, 20, 30]:
    b = (cov >= t_mark).mean() * 100
    ax3.axvline(t_mark, color='gray', ls=':', alpha=0.5)
    ax3.text(t_mark + 0.5, b - 5, f'{t_mark}×: {b:.1f}%', fontsize=8)
ax3.set_xlabel('Coverage threshold')
ax3.set_ylabel('% reference covered at threshold')
ax3.set_title('C. Breadth of coverage curve')

# Panel D: Flagstat summary bar
ax4 = axes[1, 1]
categories = ['Total', 'Mapped', 'Properly\npaired', 'Duplicates', 'Singletons']
values = [
    stats['total'],
    stats['mapped'],
    stats['properly_paired'],
    stats['duplicates'],
    stats['singletons'],
]
colors = ['steelblue', 'forestgreen', 'mediumseagreen', 'tomato', 'orange']
bars = ax4.bar(categories, values, color=colors, alpha=0.85)
for bar, val in zip(bars, values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val:,}', ha='center', fontsize=8)
ax4.set_ylabel('Number of reads')
ax4.set_title('D. samtools flagstat summary')

plt.suptitle('Module 09: Alignment QC Statistics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('alignment_stats.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. A WGS run reports: 50M reads, 47M mapped (94%), 8M duplicates (16%), mean
   coverage 28×. Is this a pass or fail for 30× WGS? What are the specific issues?
2. Implement a function `gc_bias_profile(records, ref_sequence, window=100)` that
   computes coverage vs. GC content for windows of the reference.
3. What is the expected variance of coverage at mean depth C under the Poisson model?
   How does the observed CV compare if GC bias is present?

---
## Step 10 — Quiz

1. What does `samtools flagstat` report for a run with 5% unmapped reads?
2. Why filter on MAPQ≥20 before variant calling?
3. What is the difference between breadth and depth of coverage?
4. A sample has 30× mean coverage but only 70% breadth at 20×. What does this imply?
5. What causes GC bias in Illumina sequencing?

---
## Step 12 — Reflection

> *[In one paragraph: explain to a collaborator why a 30× WGS run with 40%
> duplicate rate is effectively a 18× run, and why that changes the analysis plan.]*

---
*Next: `06_duplicate_marking_and_bqsr.ipynb`*